# Using DCCEEW project buckets <img align="right" src="../resources/csiro_easi_logo.png">

DCCEEW has a **project** bucket available for each **workspace**. At the time of writing they follow the same naming convention **<workspace>-data**
    dcceew-eds => "dcceew-eds-data",
    dcceew-veg => "dcceew-veg-data",
    dcceew-aad => "dcceew-aad-data",
    dcceew-rs  => "dcceew-rs-data",
    dcceew-t2b => "dcceew-t2b-data"

There is also an all access bucket **dcceew-epp-data** where all users have r/w access, intended for sharing data.

**Project** buckets are available to any user with access to that workspace. A project bucket can exist in another AWS account and be cross-linked to DCCEEW EASI. An admin will assign users to a "project" group, which will enable their access to the bucket(s). Files in a project bucket are subject to the bucket owner's life cycle rules, administration and costs. If a user is a member of multiple workspaces, then they can access all their project buckets regardless of which workspace the selected for their session at start up. Workspaces are use for resource tracking (compute, memory, etc). Project group membership is used to allow access by users.

Glossary:
- S3 storage items are called **objects**. Typically these are files but they could be any blob of data.
- An **object**'s name is its **key**. The **key** can be [just about any string](https://docs.aws.amazon.com/AmazonS3/latest/userguide/object-keys.html). Typically we include a `/` in the key to make it look like a directory path, which we're familiar with from regular file systems.

There are two AWS APIs that can be used to read/write to a **scratch** or **project** bucket. Examples for both are given in this notebook.
- [AWS CLI](https://docs.aws.amazon.com/cli/latest/userguide/cli-services-s3-commands.html) - linux program (use in terminal)
- [boto3](https://boto3.amazonaws.com/v1/documentation/api/latest/index.html) - python library (use in code)

We show *writing* first so that you add a test file for the *reading* section.

The examples use "dcceew-epp-data" since everyone has access but you can use any bucket you have access to (and should to test it. Try one you don't have access to as well to see what the errors look like)

- [Writing](#Writing)
   - [Select a test file](#Select-a-test-file)
   - [Upload a file](#Upload-a-file)
- [Reading](#Reading)
   - [List objects](#List-objects)
   - [Read a file directly](#Read-a-file-directly)
   - [Copy a file to local](#Copy-a-file-to-local)

## Imports and setup

In [2]:
import sys, os
import boto3
from datetime import datetime as dt

# EASI tools
import git
repo = git.Repo('.', search_parent_directories=True).working_tree_dir
if repo not in sys.path: sys.path.append(repo)


In [3]:
client = boto3.client('s3')


bucket = "dcceew-epp-data"

In [4]:
# Optional, for parallel uploads and downloads of large files
# Add a (..., Config=config) parameter to the relevant upload and download functions

# from boto3.s3.transfer import TransferConfig
# config = TransferConfig(
#     multipart_threshold = 1024 * 25,
#     max_concurrency = 10,
#     multipart_chunksize = 1024 * 25,
#     use_threads = True
# )

## Writing

For the Write test we will grab your unique UserID - this is purely so the s3 prefix (the path to the object) is unique per user so you don't clash with others when running this notebook.
In practice you would use what ever storage prefix makes sense for your project team to organise data (note this differs from **scratch** which __enforces__ the use of UserID so user scratch files don't conflict by default.

### User ID

In [5]:
%%bash

userid=`aws sts get-caller-identity --query 'UserId' | sed 's/["]//g'`
echo $userid

AROAZ6PFZYT4B4C7MNRHV:robotmcgregor


In [6]:
userid = boto3.client('sts').get_caller_identity()['UserId']
print(userid)

AROAZ6PFZYT4B4C7MNRHV:robotmcgregor


### Select a test file

For use in this notebook.

In [7]:
testfile = '/home/jovyan/test-file.txt'

In [8]:
%%bash -s "$testfile"
 
testfile=$1
touch $testfile
ls -l $testfile

-rw-r--r--. 1 jovyan users 0 Feb  9 23:53 /home/jovyan/test-file.txt


### Upload a file

In [9]:
%%bash -s "$bucket" "$userid" "$testfile"

bucket=$1
userid=$2
testfile=$3

aws s3 cp ${testfile} s3://${bucket}/${userid}/

upload: ../../../test-file.txt to s3://dcceew-epp-data/AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/test-file.txt


In [10]:
target = testfile.split('/')[-1]
try:
    print(f'upload: {testfile} to s3://{bucket}/{userid}/{target}')
    r = client.upload_file(testfile, bucket, f'{userid}/{target}')
    print('Success.')
except Exception as e:
    print(e)
    print('Failed.')

upload: /home/jovyan/test-file.txt to s3://dcceew-epp-data/AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/test-file.txt
Success.


## Reading

### List objects

The `boto3.list_objects_v2` function will return at most 1000 keys. Two options are shown here.
1. Basic use of `list_objects_v2`
2. Paginated list objects, for potentially >1000 keys

In [ ]:
%%bash -s "$bucket" "$userid"

bucket=$1
userid=$2

aws s3 ls s3://${bucket}/${userid}/

In [ ]:
# Basic use of list_objects_v2

response = client.list_objects_v2(Bucket=bucket, Prefix=f'{userid}/')

# from pprint import pprint
# pprint(response)

# List each key with its last modified time stamp
if 'Contents' in response:
    for c in response['Contents']:
        key = c['Key']
        lastmodified = c['LastModified'].strftime('%Y-%d-%m %H:%M:%S')
        size = c['Size']
        print(f'{lastmodified}\t{size} {key}')

In [ ]:
# Paginated list objects, for potentially >1000 keys

paginator = client.get_paginator('list_objects_v2')
page_iterator = paginator.paginate(Bucket=bucket, Prefix=f'{userid}/')

for response in page_iterator:
    if 'Contents' in response:
        for c in response['Contents']:
            key = c['Key']
            lastmodified = c['LastModified'].strftime('%Y-%d-%m %H:%M:%S')
            psize = c['Size']
            print(f'{lastmodified}\t{size} {key}')

### Read a file directly

Many data reading packages can read a file from an *s3://bucket/key* path into memory. Examples include:
- `rasterio` and `rioxarray`
- `gdal`

For packages that can not read from an S3 path, first copy the file to your home directory or a temporary directory (e.g., dask workers). Then read the file with a normal file path.

### Copy a file to local

In [ ]:
%%bash -s "$bucket" "$userid" "$testfile"

bucket=$1
userid=$2
testfile=$3

source=`basename $testfile`
aws s3 cp s3://${bucket}/${userid}/${source} ${testfile}
ls -l $testfile

In [ ]:
source = testfile.split('/')[-1]
try:
    print(f'download: s3://{bucket}/{userid}/{source} to {testfile}')
    r = client.download_file(bucket, f'{userid}/{source}', testfile)
    print('Success.')
except Exception as e:
    print(e)
    print('Failed.')

# Clean up

We can use the aws cli to clean up by recursively removing the files.
It has a --dry-run option so you can see what action it will take prior to letting it loose for real.

The list buckets at the end should toss an error given the prefix will no longer exist if the deletion is successful.

In [ ]:
%%bash -s "$bucket" "$userid"

bucket=$1
userid=$2

aws s3 rm s3://${bucket}/${userid}  --recursive --dryrun

In [ ]:
%%bash -s "$bucket" "$userid"

bucket=$1
userid=$2

aws s3 rm s3://${bucket}/${userid}  --recursive

In [ ]:
%%bash -s "$bucket" "$userid"

bucket=$1
userid=$2

aws s3 ls s3://${bucket}/${userid}/